# SparkSession

SparkSession is automatically created when you start up a Notebook (e.g. Zeppelin, Databricks)

<img src="https://i.imgur.com/5Ai45fb.jpg" width=500px>

In [ ]:
%spark
//Scala SparkSession
spark

res2: org.apache.spark.sql.SparkSession = org.apache.spark.sql.SparkSession@2a285fd4


In [ ]:
%spark.pyspark
#PySpark SparkSession
spark

SparkSession - hive 
 
 
 SparkContext 

 Spark UI 

 
 Version 
 v3.3.2 
 Master 
 yarn 
 AppName 
 spark-shared_process

# Show DataFrame

`df.show()` is the Spark native API that displays data but it's not pretty. 

`z.show(df)` is a Zeppelin build-in feature that allows you to show a df result in a pretty way

In [ ]:
%spark.pyspark

#List all hive tables in a df
tables_df = spark.sql("show tables")

In [ ]:
%spark.pyspark

tables_df.show()

+---------+---------------+-----------+
|namespace|      tableName|isTemporary|
+---------+---------------+-----------+
|  default|wdi_csv_parquet|      false|
+---------+---------------+-----------+



In [ ]:
%spark.pyspark
z.show(tables_df)

namespace,tableName,isTemporary
default,wdi_csv_parquet,false


# Spark SQL vs Dataframe

`%sql` is the Spark SQL interpreter

`%spark.pyspark` is the PySpark interpreter

`%spark` is the Spark Scala interpreter

In [ ]:
%sql

select count(1) from wdi_csv_parquet

count(1)
21759413


In [ ]:
%spark.pyspark

#Read Hive data to a df (this is lazy)
wdi_df = spark.sql("SELECT * from wdi_csv_parquet")
#Persist df in memory for fast futuer access
wdi_df = wdi_df.cache()
wdi_df.printSchema()

#Spark action is eager
z.show(wdi_df.count())

root
 |-- year: integer (nullable = true)
 |-- countryname: string (nullable = true)
 |-- countrycode: string (nullable = true)
 |-- indicatorname: string (nullable = true)
 |-- indicatorcode: string (nullable = true)
 |-- indicatorvalue: float (nullable = true)

21759413


# Show Historical GDP for Canada

- Re-write the hive query (left cell) using PySpark df

In [ ]:
%sql
SELECT year, IndicatorValue as GDP
FROM wdi_csv_parquet
WHERE indicatorCode = 'NY.GDP.MKTP.KD.ZG' and countryName = 'Canada'
ORDER BY year


year,GDP
1960,0.0
1961,3.1632917
1962,7.1167793
1963,5.181776
1964,6.6994567
1965,6.636545
1966,6.584965
1967,2.9153104
1968,5.295342
1969,5.2600007


In [ ]:
%spark.pyspark
from pyspark.sql import functions as F

wdi_canada_df = spark.table("wdi_csv_parquet") \
    .filter(
        (F.col("indicatorCode") == "NY.GDP.MKTP.KD.ZG") &
        (F.col("countryName") == "Canada")
    ) \
    .select("year", F.col("IndicatorValue").alias("GDP")) \
    .orderBy("year")

wdi_canada_df.show()

+----+---------+
|year|      GDP|
+----+---------+
|1960|      0.0|
|1961|3.1632917|
|1962|7.1167793|
|1963| 5.181776|
|1964|6.6994567|
|1965| 6.636545|
|1966| 6.584965|
|1967|2.9153104|
|1968| 5.295342|
|1969|5.2600007|
|1970|3.2556136|
|1971|4.1176577|
|1972|5.4458556|
|1973| 6.964203|
|1974|3.6909876|
|1975|1.8229731|
|1976|5.1993027|
|1977|3.4582305|
|1978|3.9535913|
|1979|3.8049228|
+----+---------+
only showing top 20 rows



# Show GDP for Each County and Sort By Year

- Re-write the hive query (left cell) using PySpark df  
    - hint: you can create multiple DFs

In [ ]:
%sql
SELECT countryname,
       year,
       indicatorcode,
       indicatorvalue
FROM wdi_csv_parquet
WHERE indicatorcode = 'NY.GDP.MKTP.KD.ZG'
DISTRIBUTE BY countryname
SORT BY countryname, year


countryname,year,indicatorcode,indicatorvalue
Afghanistan,1960,NY.GDP.MKTP.KD.ZG,0.0
Afghanistan,1961,NY.GDP.MKTP.KD.ZG,0.0
Afghanistan,1962,NY.GDP.MKTP.KD.ZG,0.0
Afghanistan,1963,NY.GDP.MKTP.KD.ZG,0.0
Afghanistan,1964,NY.GDP.MKTP.KD.ZG,0.0
Afghanistan,1965,NY.GDP.MKTP.KD.ZG,0.0
Afghanistan,1966,NY.GDP.MKTP.KD.ZG,0.0
Afghanistan,1967,NY.GDP.MKTP.KD.ZG,0.0
Afghanistan,1968,NY.GDP.MKTP.KD.ZG,0.0
Afghanistan,1969,NY.GDP.MKTP.KD.ZG,0.0


× Output is truncated to 1000 rows. Learn more about zeppelin.spark.maxResult

In [ ]:
%spark.pyspark
from pyspark.sql import functions as F

gdp_df = spark.table("wdi_csv_parquet") \
    .filter(F.col("indicatorcode") == "NY.GDP.MKTP.KD.ZG") \
    .select("countryname", "year", "indicatorcode", "indicatorvalue") \
    .repartition("countryname") \
    .sortWithinPartitions("countryname", "year")

gdp_df.show(20)

+-----------+----+-----------------+--------------+
|countryname|year|    indicatorcode|indicatorvalue|
+-----------+----+-----------------+--------------+
|Afghanistan|1960|NY.GDP.MKTP.KD.ZG|           0.0|
|Afghanistan|1961|NY.GDP.MKTP.KD.ZG|           0.0|
|Afghanistan|1962|NY.GDP.MKTP.KD.ZG|           0.0|
|Afghanistan|1963|NY.GDP.MKTP.KD.ZG|           0.0|
|Afghanistan|1964|NY.GDP.MKTP.KD.ZG|           0.0|
|Afghanistan|1965|NY.GDP.MKTP.KD.ZG|           0.0|
|Afghanistan|1966|NY.GDP.MKTP.KD.ZG|           0.0|
|Afghanistan|1967|NY.GDP.MKTP.KD.ZG|           0.0|
|Afghanistan|1968|NY.GDP.MKTP.KD.ZG|           0.0|
|Afghanistan|1969|NY.GDP.MKTP.KD.ZG|           0.0|
|Afghanistan|1970|NY.GDP.MKTP.KD.ZG|           0.0|
|Afghanistan|1971|NY.GDP.MKTP.KD.ZG|           0.0|
|Afghanistan|1972|NY.GDP.MKTP.KD.ZG|           0.0|
|Afghanistan|1973|NY.GDP.MKTP.KD.ZG|           0.0|
|Afghanistan|1974|NY.GDP.MKTP.KD.ZG|           0.0|
|Afghanistan|1975|NY.GDP.MKTP.KD.ZG|           0.0|
|Afghanistan

# Find the highest GDP for each country

- Re-write the hive query (left cell) using PySpark df

In [ ]:
%sql

SELECT wdi_csv_parquet.indicatorvalue AS value, 
       wdi_csv_parquet.year           AS year, 
       wdi_csv_parquet.countryname    AS country 
FROM   (SELECT Max(indicatorvalue) AS ind, 
               countryname 
        FROM   wdi_csv_parquet 
        WHERE  indicatorcode = 'NY.GDP.MKTP.KD.ZG' 
               AND indicatorvalue <> 0 
        GROUP  BY countryname) t1 
       INNER JOIN wdi_csv_parquet 
               ON t1.ind = wdi_csv_parquet.indicatorvalue 
                  AND t1.countryname = wdi_csv_parquet.countryname


value,year,country
17.505634,2000,Timor-Leste
5.2550063,1990,Germany
8.809809,1986,Uruguay
7.1668916,1970,Australia
33.62937,2004,Chad
6.645474,1997,Croatia
20.286634,1985,Mali
8.755049,1973,Latin America & the Caribbean (IDA & IBRD countries)
8.643095,1965,Netherlands
14.040795,2003,Armenia


In [ ]:
%spark.pyspark
from pyspark.sql import functions as F

# Base filtered dataframe
df = spark.table("wdi_csv_parquet") \
    .filter(
        (F.col("indicatorcode") == "NY.GDP.MKTP.KD.ZG") &
        (F.col("indicatorvalue") != 0)
    ).alias("a")
# Subquery: max GDP per country
max_df = df.groupBy("countryname") \
    .agg(F.max("indicatorvalue").alias("ind")) \
    .alias("b")


result_df = df.join(
    max_df,
    (F.col("a.countryname") == F.col("b.countryname")) &
    (F.col("a.indicatorvalue") == F.col("b.ind"))
).select(
    F.col("a.indicatorvalue").alias("value"),
    F.col("a.year").alias("year"),
    F.col("a.countryname").alias("country")
)

result_df.limit(100).show()

+---------+----+--------------------+
|    value|year|             country|
+---------+----+--------------------+
|  9.08357|2010|          South Asia|
| 33.62937|2004|                Chad|
| 8.394322|2004| Lower middle income|
|14.036278|2013|            Paraguay|
| 8.618244|2007| Low & middle income|
|6.8927703|1962|Heavily indebted ...|
| 6.645997|1964|               World|
| 8.920504|1976|             Senegal|
|21.200697|1962|    Congo, Dem. Rep.|
| 6.821457|1964|              Sweden|
|19.182642|1994|          Cabo Verde|
|13.160728|1970|East Asia & Pacif...|
|45.302753|1974|            Kiribati|
| 9.378625|2007|Least developed c...|
| 6.473487|2007|      Macedonia, FYR|
|11.360282|1964|              Guyana|
| 8.920647|1973|         Philippines|
|6.6795015|1984|Pacific island sm...|
| 21.22141|1994|             Eritrea|
|6.5584636|1985|               Tonga|
+---------+----+--------------------+
only showing top 20 rows



In [ ]:
%spark.pyspark
